# Trace Data Viewer GUI

Launch the local Dash GUI for browsing CKII merged aligned traces, spikes, complex bursts, plateaus, and recalculated bad-SNR periods.

In [1]:
from pathlib import Path

repo_root = Path('/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline')
analysis_root = repo_root / 'miniVI_PlaceCell_analysis_V4'
data_root = analysis_root / 'data'

host = '127.0.0.1'
port = 8053
debug = False

print(f'Data root: {data_root}')

Data root: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data


In [2]:
# Export full-duration raw pooled place-cell traces into two category-specific HTML files.
# No downsampling is applied, so spike detection can be evaluated at full resolution.
# Hovering over the main trace reports the recalculated time-varying SNR at that frame.
# Bad-SNR shading is shown by default; Vm, plateaus, and complex-burst spans are hidden by default but available from the legend.
import sys

if str(analysis_root) not in sys.path:
    sys.path.insert(0, str(analysis_root))

from dash_data_viewer_app.figure_builder import plot_pooled_place_cells_by_category_html

pooled_place_cells_output_dir = analysis_root / 'figures' / 'CKII_pooled'
pooled_place_cell_outputs = plot_pooled_place_cells_by_category_html(
    data_root=data_root,
    output_dir=pooled_place_cells_output_dir,
    snr_threshold=3.5,
    min_good_minutes=5.0,
    trace_max_points=None,
    optional_layers_visible=True,
    bad_epochs_visible=True,
    include_plotlyjs=True,
    segment_duration_s=120.0,
    show_legend=False,
)

for category, result in pooled_place_cell_outputs.items():
    print(f"{category}: saved {result['path']} ({result['n_cells']} cells, {result.get('n_segments', 1)} segment(s))")


CS+ PLC: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_place_cells_CSplus_PLC_trace_viewer.html (14 cells, 10 segment(s))
CS- PLC: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_place_cells_CSminus_PLC_trace_viewer.html (15 cells, 10 segment(s))


In [3]:
# Export one combined pooled place-cell HTML with simple numeric row labels.
# The accompanying CSV maps each row number to the original animal/cell identity.
import csv
import sys
from pathlib import Path

if str(analysis_root) not in sys.path:
    sys.path.insert(0, str(analysis_root))

from dash_data_viewer_app.figure_builder import plot_pooled_place_cells_all_animals_html


def _write_numbered_identity_csv(stats_rows, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'row_number',
        'cell_identity',
        'animal_id',
        'cell_number',
        'cell_idx',
        'category',
        'time of good spike detection (sec)',
    ]
    with csv_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row_number, row in enumerate(stats_rows, start=1):
            cell_idx = int(row.get('cell_idx', -1))
            animal_id = row.get('animal_id', 'unknown')
            cell_identity = f"{animal_id} | Cell {cell_idx + 1} (idx {cell_idx})" if cell_idx >= 0 else str(animal_id)
            writer.writerow(
                {
                    'row_number': row_number,
                    'cell_identity': cell_identity,
                    'animal_id': animal_id,
                    'cell_number': cell_idx + 1 if cell_idx >= 0 else '',
                    'cell_idx': cell_idx if cell_idx >= 0 else '',
                    'category': row.get('category', 'Unknown'),
                    'time of good spike detection (sec)': '',
                }
            )
    return csv_path


combined_pooled_output_path = analysis_root / 'figures' / 'CKII_pooled' / 'pooled_place_cells_combined_numbered_trace_viewer.html'
combined_fig, combined_stats_rows, combined_html_path = plot_pooled_place_cells_all_animals_html(
    data_root=data_root,
    output_html=combined_pooled_output_path,
    snr_threshold=3.5,
    min_good_minutes=5.0,
    trace_max_points=None,
    category_filter={'CS+ PLC', 'CS- PLC'},
    optional_layers_visible=True,
    bad_epochs_visible=True,
    include_plotlyjs=True,
    segment_duration_s=120.0,
    show_legend=False,
    y_label_mode='number',
)
combined_identity_csv_path = _write_numbered_identity_csv(
    combined_stats_rows,
    combined_pooled_output_path.with_name('pooled_place_cells_combined_numbered_identity_map.csv'),
)

print(f"Combined PLC HTML: saved {combined_html_path} ({len(combined_stats_rows)} cells)")
print(f"Combined PLC identity CSV: saved {combined_identity_csv_path}")


Combined PLC HTML: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_place_cells_combined_numbered_trace_viewer.html (29 cells)
Combined PLC identity CSV: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_place_cells_combined_numbered_identity_map.csv


In [3]:
# Export combined non-place-cell HTML in numbered chunks.
# The two HTML files share continuous row numbering, and one CSV maps all rows to identities.
import csv
import sys
from pathlib import Path

if str(analysis_root) not in sys.path:
    sys.path.insert(0, str(analysis_root))

from dash_data_viewer_app.data_io import discover_animals, load_animal
from dash_data_viewer_app.figure_builder import plot_pooled_place_cells_all_animals_html


def _write_numbered_identity_csv(stats_rows, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'row_number',
        'cell_identity',
        'animal_id',
        'cell_number',
        'cell_idx',
        'category',
        'time of good spike detection (sec)',
    ]
    with csv_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row_number, row in enumerate(stats_rows, start=1):
            cell_idx = int(row.get('cell_idx', -1))
            animal_id = row.get('animal_id', 'unknown')
            cell_identity = f"{animal_id} | Cell {cell_idx + 1} (idx {cell_idx})" if cell_idx >= 0 else str(animal_id)
            writer.writerow(
                {
                    'row_number': row_number,
                    'cell_identity': cell_identity,
                    'animal_id': animal_id,
                    'cell_number': cell_idx + 1 if cell_idx >= 0 else '',
                    'cell_idx': cell_idx if cell_idx >= 0 else '',
                    'category': row.get('category', 'Unknown'),
                    'time of good spike detection (sec)': '',
                }
            )
    return csv_path


animal_ids = discover_animals(data_root)
non_place_cell_keys = []
for animal_id in animal_ids:
    animal = load_animal(data_root, animal_id)
    place_mask = getattr(animal, 'place_cell_mask', [])
    for cell_idx in range(animal.n_cells):
        is_place_cell = bool(cell_idx < len(place_mask) and place_mask[cell_idx])
        if not is_place_cell:
            non_place_cell_keys.append((animal.animal_id, int(cell_idx)))

non_place_output_dir = analysis_root / 'figures' / 'CKII_pooled'
non_place_batches = [non_place_cell_keys[:25], non_place_cell_keys[25:]]
non_place_stats_rows = []
non_place_html_paths = []
for batch_idx, batch_keys in enumerate(non_place_batches, start=1):
    if not batch_keys:
        continue
    start_number = 1 if batch_idx == 1 else 26
    end_number = start_number + len(batch_keys) - 1
    non_place_output_path = non_place_output_dir / f'pooled_non_place_cells_combined_numbered_{batch_idx:02d}_cells_{start_number}-{end_number}_trace_viewer.html'
    _, batch_stats_rows, batch_html_path = plot_pooled_place_cells_all_animals_html(
        data_root=data_root,
        output_html=non_place_output_path,
        animal_ids=animal_ids,
        snr_threshold=3.5,
        min_good_minutes=5.0,
        trace_max_points=None,
        cell_selection='non_place',
        cell_key_filter=set(batch_keys),
        optional_layers_visible=True,
        bad_epochs_visible=True,
        include_plotlyjs=True,
        segment_duration_s=120.0,
        show_legend=False,
        y_label_mode='number',
    )
    non_place_stats_rows.extend(batch_stats_rows)
    non_place_html_paths.append((batch_idx, start_number, end_number, batch_html_path))

non_place_identity_csv_path = _write_numbered_identity_csv(
    non_place_stats_rows,
    non_place_output_dir / 'pooled_non_place_cells_combined_numbered_identity_map.csv',
)

for batch_idx, start_number, end_number, batch_html_path in non_place_html_paths:
    print(f"Non-place-cell HTML {batch_idx}: saved {batch_html_path} (rows {start_number}-{end_number})")
print(f"Combined non-place-cell identity CSV: saved {non_place_identity_csv_path} ({len(non_place_stats_rows)} cells)")


Non-place-cell HTML 1: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_non_place_cells_combined_numbered_01_cells_1-25_trace_viewer.html (rows 1-25)
Non-place-cell HTML 2: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_non_place_cells_combined_numbered_02_cells_26-47_trace_viewer.html (rows 26-47)
Combined non-place-cell identity CSV: saved /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pooled/pooled_non_place_cells_combined_numbered_identity_map.csv (47 cells)


In [4]:
import os
import signal
import subprocess
import sys
import time
import webbrowser

app_script = analysis_root / 'dash_data_viewer_app' / 'app.py'
if not app_script.is_file():
    raise FileNotFoundError(f'Cannot find app.py at: {app_script}')
if not data_root.is_dir():
    raise FileNotFoundError(f'Cannot find data root: {data_root}')

discovered_animals = sorted(
    child.name
    for child in data_root.iterdir()
    if child.is_dir()
    and ((child / 'merged_aligned_data.pkl').exists() or (child / 'merged_aligned_data_CS.pkl').exists())
)
print(f'Discovered {len(discovered_animals)} datasets:')
for animal in discovered_animals:
    print(f'  {animal}')

def _trace_viewer_pids_to_stop():
    pids = set()
    if '_trace_viewer_proc' in globals() and _trace_viewer_proc is not None and _trace_viewer_proc.poll() is None:
        pids.add(int(_trace_viewer_proc.pid))

    # Stop whatever is holding this viewer port, including app instances from older notebook kernels.
    port_result = subprocess.run(
        ['lsof', f'-tiTCP:{int(port)}', '-sTCP:LISTEN'],
        text=True,
        capture_output=True,
        check=False,
    )
    for token in port_result.stdout.split():
        try:
            pid = int(token)
        except ValueError:
            continue
        pids.add(pid)

    # Stop old Trace Data Viewer processes that may have been started on another port.
    app_result = subprocess.run(
        ['pgrep', '-f', 'dash_data_viewer_app/app.py'],
        text=True,
        capture_output=True,
        check=False,
    )
    for token in app_result.stdout.split():
        try:
            pid = int(token)
        except ValueError:
            continue
        if pid != os.getpid():
            pids.add(pid)

    return sorted(pid for pid in pids if pid != os.getpid())

def _pid_is_alive(pid):
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    return True

def _stop_existing_trace_viewers():
    pids = _trace_viewer_pids_to_stop()
    if not pids:
        print('No existing Trace Data Viewer processes found.')
        return

    print('Stopping existing Trace Data Viewer process(es): ' + ', '.join(str(pid) for pid in pids))
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except PermissionError as exc:
            print(f'Could not stop PID {pid}: {exc}')

    deadline = time.time() + 5.0
    remaining = [pid for pid in pids if _pid_is_alive(pid)]
    while remaining and time.time() < deadline:
        time.sleep(0.2)
        remaining = [pid for pid in remaining if _pid_is_alive(pid)]

    for pid in remaining:
        try:
            os.kill(pid, signal.SIGKILL)
            print(f'Force-stopped PID {pid}.')
        except ProcessLookupError:
            pass
        except PermissionError as exc:
            print(f'Could not force-stop PID {pid}: {exc}')
    time.sleep(0.5)

browser_host = '127.0.0.1' if host in ('0.0.0.0', '::', '') else host
url = f'http://{browser_host}:{int(port)}'

_stop_existing_trace_viewers()
if '_trace_viewer_log' in globals() and _trace_viewer_log is not None and not _trace_viewer_log.closed:
    _trace_viewer_log.close()

cmd = [
    sys.executable,
    str(app_script),
    '--data-root', str(data_root),
    '--host', str(host),
    '--port', str(int(port)),
    '--no-browser',
]
if debug:
    cmd.append('--debug')

_trace_viewer_log_path = analysis_root / f'trace_data_viewer_{int(port)}.log'
_trace_viewer_log = open(_trace_viewer_log_path, 'w')
_trace_viewer_proc = subprocess.Popen(
    cmd,
    cwd=str(analysis_root),
    stdout=_trace_viewer_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

time.sleep(1.0)
if _trace_viewer_proc.poll() is not None:
    _trace_viewer_log.flush()
    log_text = _trace_viewer_log_path.read_text(errors='replace')
    print(log_text[-4000:])
    raise RuntimeError(f'Trace Data Viewer failed to start. See {_trace_viewer_log_path}')
_ = webbrowser.open(url)
print(f'Trace Data Viewer running at {url}')
print(f'Log file: {_trace_viewer_log_path}')

Discovered 7 datasets:
  CKII_pAce21_PR_20250806
  CKII_pAce38_PX_20251126
  CKII_pAce45_PX_20260118
  CKII_pAce46_PR_20260222
  CKII_pAce47_PX_20260128
  CKII_pAce50_PRL_20260317
  CKII_pAce54_PR_20260506
Stopping existing Trace Data Viewer process(es): 94717, 94722
Trace Data Viewer running at http://127.0.0.1:8053
Log file: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/trace_data_viewer_8053.log


## Optional: share the GUI with Cloudflare Tunnel

Run this after the Dash launch cell if you want to give someone outside your local network a temporary public URL. The URL changes each time the tunnel is restarted.


In [3]:
import os
import re
import shutil
import signal
import subprocess
import time
from pathlib import Path

cloudflared_path = shutil.which('cloudflared')
if cloudflared_path is None:
    for candidate in ('/opt/homebrew/bin/cloudflared', '/usr/local/bin/cloudflared'):
        if Path(candidate).is_file():
            cloudflared_path = candidate
            break
if cloudflared_path is None:
    raise FileNotFoundError(
        'cloudflared is not installed or not on PATH. Install it in Terminal with: brew install cloudflared'
    )

if '_cloudflare_tunnel_proc' in globals() and _cloudflare_tunnel_proc is not None and _cloudflare_tunnel_proc.poll() is None:
    _cloudflare_tunnel_proc.terminate()
    try:
        _cloudflare_tunnel_proc.wait(timeout=5)
        print('Stopped previous Cloudflare Tunnel from this notebook kernel.')
    except subprocess.TimeoutExpired:
        os.kill(_cloudflare_tunnel_proc.pid, signal.SIGKILL)
        _cloudflare_tunnel_proc.wait(timeout=5)
        print('Force-stopped previous Cloudflare Tunnel from this notebook kernel.')

if '_cloudflare_tunnel_log' in globals() and _cloudflare_tunnel_log is not None and not _cloudflare_tunnel_log.closed:
    _cloudflare_tunnel_log.close()

browser_host = '127.0.0.1' if host in ('0.0.0.0', '::', '') else host
local_url = f'http://{browser_host}:{int(port)}'
_cloudflare_tunnel_log_path = analysis_root / f'cloudflare_tunnel_{int(port)}.log'
_cloudflare_tunnel_log = open(_cloudflare_tunnel_log_path, 'w')
_cloudflare_tunnel_proc = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--url', local_url],
    cwd=str(analysis_root),
    stdout=_cloudflare_tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

url_pattern = re.compile(r'https://[A-Za-z0-9-]+\.trycloudflare\.com')
_cloudflare_tunnel_url = None
deadline = time.time() + 30.0
while time.time() < deadline:
    if _cloudflare_tunnel_proc.poll() is not None:
        _cloudflare_tunnel_log.flush()
        log_text = _cloudflare_tunnel_log_path.read_text(errors='replace')
        print(log_text[-4000:])
        raise RuntimeError(f'Cloudflare Tunnel exited early. See {_cloudflare_tunnel_log_path}')

    _cloudflare_tunnel_log.flush()
    log_text = _cloudflare_tunnel_log_path.read_text(errors='replace')
    match = url_pattern.search(log_text)
    if match:
        _cloudflare_tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if not _cloudflare_tunnel_url:
    _cloudflare_tunnel_log.flush()
    log_text = _cloudflare_tunnel_log_path.read_text(errors='replace')
    print(log_text[-4000:])
    raise TimeoutError(f'Cloudflare Tunnel did not print a trycloudflare.com URL within 30 seconds. See {_cloudflare_tunnel_log_path}')

print(f'Cloudflare Tunnel running for local app: {local_url}')
print(f'Send this URL to your friend: {_cloudflare_tunnel_url}')
print(f'Tunnel log file: {_cloudflare_tunnel_log_path}')


Cloudflare Tunnel running for local app: http://127.0.0.1:8053
Send this URL to your friend: https://arrive-log-son-poker.trycloudflare.com
Tunnel log file: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/cloudflare_tunnel_8053.log


In [4]:
# Stop the GUI process started from this notebook, if needed.
if '_trace_viewer_proc' in globals() and _trace_viewer_proc is not None and _trace_viewer_proc.poll() is None:
    _trace_viewer_proc.terminate()
    print('Stopped Trace Data Viewer.')
else:
    print('Trace Data Viewer is not running from this kernel.')

# Stop the Cloudflare Tunnel process started from this notebook, if needed.
if '_cloudflare_tunnel_proc' in globals() and _cloudflare_tunnel_proc is not None and _cloudflare_tunnel_proc.poll() is None:
    _cloudflare_tunnel_proc.terminate()
    print('Stopped Cloudflare Tunnel.')
else:
    print('Cloudflare Tunnel is not running from this kernel.')


Stopped Trace Data Viewer.
Stopped Cloudflare Tunnel.
